# Moduł 5: Wstęp do uczenia maszynowego

**Uczenie maszynowe (ML)** to dziedzina AI, w której komputer **uczy się z danych** zamiast być programowany jawnie.

## Spis treści
1. Typy uczenia
2. Podział danych: train / val / test
3. Metryki ewaluacji
4. Overfitting i underfitting
5. Walidacja krzyżowa (cross-validation)
6. Przygotowanie danych
7. Redukcja wymiarowości (PCA)
8. **Zadania**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.datasets import load_iris, make_classification
sns.set_style("whitegrid")

## 1. Typy uczenia maszynowego

### Uczenie nadzorowane (supervised)
- Mamy zbiór danych $\mathcal{D} = \{(x_i, y_i)\}_{i=1}^{n}$ — pary (dane, etykieta)
- Model uczy się funkcji $f: \mathcal{X} \to \mathcal{Y}$ takiej, że $f(x) \approx y$
- **Klasyfikacja:** $y \in \{1, 2, \dots, K\}$ — klasy dyskretne (kot/pies, spam/nie-spam)
- **Regresja:** $y \in \mathbb{R}$ — wartości ciągłe (cena domu, temperatura)

### Uczenie nienadzorowane (unsupervised)
- Tylko dane $\mathcal{D} = \{x_i\}_{i=1}^{n}$, **bez etykiet**
- Model szuka struktury/wzorców w danych
- **Klasteryzacja:** partycja zbioru na grupy $C_1, C_2, \dots, C_K$ minimalizująca odległości wewnątrz-grupowe
- **Redukcja wymiarów:** odwzorowanie $f: \mathbb{R}^d \to \mathbb{R}^k$ gdzie $k \ll d$, zachowujące strukturę danych

### Uczenie ze wzmocnieniem (reinforcement learning)
- Agent podejmuje akcje $a_t$ w stanie $s_t$, przechodząc do stanu $s_{t+1}$
- Otrzymuje nagrodę $r_t$ za każdą akcję
- Cel: znaleźć politykę $\pi(a|s)$ maksymalizującą **skumulowaną zdyskontowaną nagrodę**:

$$G_t = \sum_{k=0}^{\infty} \gamma^k r_{t+k+1}$$

gdzie $\gamma \in [0, 1]$ to współczynnik dyskonta (jak bardzo cenimy przyszłe nagrody).

## 2. Podział danych: Train / Validation / Test

**Dlaczego dzielimy?** Bo chcemy sprawdzić, czy model **uogólnia** na nowe, niewidziane dane.

| Zbiór | Cel | Typowy % |
|-------|-----|----------|
| **Train** | Uczenie modelu | 60-80% |
| **Validation** | Dobór hiperparametrów | 10-20% |
| **Test** | Finalna ocena | 10-20% |

**ZŁOTA ZASADA:** Zbiór testowy oglądasz **TYLKO RAZ**, na samym końcu!

In [ ]:
# Przykład podziału danych
iris = load_iris()
X, y = iris.data, iris.target

# Podział na train+val i test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
# Podział train+val na train i val
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval
)

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)")
print(f"Val:   {len(X_val)} ({len(X_val)/len(X)*100:.0f}%)")
print(f"Test:  {len(X_test)} ({len(X_test)/len(X)*100:.0f}%)")

# stratify=y zapewnia, że proporcje klas są zachowane w każdym zbiorze

## 3. Metryki ewaluacji

### Macierz pomyłek (Confusion Matrix)

Dla klasyfikacji binarnej (klasa pozytywna vs negatywna):

|  | Predicted Positive | Predicted Negative |
|--|-------------------|-------------------|
| **Actual Positive** | TP (True Positive) | FN (False Negative) |
| **Actual Negative** | FP (False Positive) | TN (True Negative) |

### Kluczowe metryki

**Accuracy** — odsetek poprawnych predykcji:

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**Precision** — z tych, które model oznaczył jako pozytywne, ile jest faktycznie pozytywnych:

$$\text{Precision} = \frac{TP}{TP + FP}$$

**Recall (Sensitivity/Czułość)** — z faktycznie pozytywnych, ile model złapał:

$$\text{Recall} = \frac{TP}{TP + FN}$$

**F1-score** — średnia harmoniczna precision i recall:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot TP}{2 \cdot TP + FP + FN}$$

**F-beta score** — uogólnienie F1, gdzie $\beta$ kontroluje wagę recall vs precision:

$$F_\beta = (1 + \beta^2) \cdot \frac{\text{Precision} \cdot \text{Recall}}{\beta^2 \cdot \text{Precision} + \text{Recall}}$$

- $\beta = 1$ → F1 (równowaga precision i recall)
- $\beta = 2$ → F2 (recall ważniejszy — np. diagnostyka medyczna)
- $\beta = 0.5$ → F0.5 (precision ważniejszy — np. filtr spamu)

**Balanced Accuracy** — średnia recall z każdej klasy (odporna na niezbalansowane klasy):

$$\text{Balanced Accuracy} = \frac{1}{K}\sum_{k=1}^{K} \text{Recall}_k$$

**ROC AUC** — pole pod krzywą ROC (Receiver Operating Characteristic):
- Krzywa ROC rysuje $\text{TPR}$ (True Positive Rate = Recall) vs $\text{FPR}$ (False Positive Rate = $\frac{FP}{FP+TN}$) dla różnych progów klasyfikacji
- AUC $\in [0.5, 1.0]$ — 0.5 = losowy klasyfikator, 1.0 = idealny
- **Interpretacja probabilistyczna:** $P(\text{score}(x^+) > \text{score}(x^-))$ — prawdopodobieństwo, że model przypisze wyższy score pozytywnej próbce niż negatywnej

### Kiedy co?
- **Accuracy** — gdy klasy są zbalansowane
- **Precision** — gdy FP jest kosztowny (np. filtr spamu — nie chcesz usuwać ważnych maili)
- **Recall** — gdy FN jest kosztowny (np. diagnostyka raka — nie chcesz pominąć chorego)
- **F1** — kompromis między precision i recall
- **Balanced Accuracy** — niezbalansowane klasy (np. 95% klasy 0, 5% klasy 1)
- **ROC AUC** — ocena ogólnej jakości rankingu (niezależna od progu)

In [ ]:
# Trening i ewaluacja KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_val)

# Metryki
print("=== METRYKI ===")
print(f"Accuracy:  {accuracy_score(y_val, y_pred):.3f}")
print(f"Precision: {precision_score(y_val, y_pred, average='macro'):.3f}")
print(f"Recall:    {recall_score(y_val, y_pred, average='macro'):.3f}")
print(f"F1-score:  {f1_score(y_val, y_pred, average='macro'):.3f}")

# Pełny raport
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_val, y_pred, target_names=iris.target_names))

# Macierz pomyłek
cm = confusion_matrix(y_val, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=iris.target_names, yticklabels=iris.target_names)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Macierz pomyłek")
plt.show()

## 4. Overfitting vs Underfitting

| Problem | Opis | Objawy |
|---------|------|--------|
| **Underfitting** | Model za prosty | Słaby wynik na train I test |
| **Overfitting** | Model "zapamiętał" dane | Świetny train, słaby test |
| **Just right** | Model uogólnia | Dobry wynik na obu |

### Bias-Variance Tradeoff — dekompozycja błędu

Błąd generalizacji modelu można rozłożyć na trzy składniki:

$$\text{MSE} = \mathbb{E}\left[(y - \hat{f}(x))^2\right] = \underbrace{\text{Bias}^2(\hat{f})}_{\text{obciążenie}} + \underbrace{\text{Var}(\hat{f})}_{\text{wariancja}} + \underbrace{\sigma^2}_{\text{szum}}$$

Gdzie:
- **Bias** $= \mathbb{E}[\hat{f}(x)] - f(x)$ — średnie odchylenie predykcji od prawdziwej wartości. Wysoki bias = underfitting (model zbyt prosty, nie oddaje złożoności danych).
- **Variance** $= \mathbb{E}\left[(\hat{f}(x) - \mathbb{E}[\hat{f}(x)])^2\right]$ — jak bardzo predykcja zmienia się zależnie od konkretnego zbioru treningowego. Wysoka wariancja = overfitting (model nadmiernie dopasowuje się do danych).
- **$\sigma^2$** — nieredukowalny szum w danych (nie da się go wyeliminować).

**Trade-off:** Zmniejszenie bias (bardziej złożony model) → zwiększenie wariancji i odwrotnie. Optymalny model balansuje oba.

### Regularyzacja — matematyczne ograniczenie złożoności

Dodajemy **karę za złożoność** do funkcji kosztu:

$$\mathcal{L}_{\text{reg}}(\theta) = \mathcal{L}(\theta) + \lambda \cdot \Omega(\theta)$$

| Typ | Kara $\Omega(\theta)$ | Efekt |
|-----|----------------------|-------|
| **L2 (Ridge)** | $\|\theta\|_2^2 = \sum \theta_i^2$ | Zmniejsza wagi (nie do zera) |
| **L1 (Lasso)** | $\|\theta\|_1 = \sum |\theta_i|$ | Zeruje nieważne wagi (selekcja cech!) |
| **Elastic Net** | $\alpha\|\theta\|_1 + (1-\alpha)\|\theta\|_2^2$ | Kombinacja obu |

$\lambda$ — siła regularyzacji: $\lambda = 0$ → brak regularyzacji, $\lambda \to \infty$ → model trywialny.

### Jak zapobiegać overfittingowi?
1. Więcej danych treningowych
2. Regularyzacja (L1, L2, Elastic Net)
3. Walidacja krzyżowa
4. Dropout (w sieciach neuronowych)
5. Early stopping
6. Data augmentation

In [ ]:
# Demonstracja overfitting/underfitting na KNN
# k=1: overfit (zapamiętuje każdy punkt)
# k=50: underfit (za dużo uśredniania)

k_values = range(1, 31)
train_scores = []
val_scores = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    train_scores.append(knn.score(X_train, y_train))
    val_scores.append(knn.score(X_val, y_val))

plt.figure(figsize=(10, 5))
plt.plot(k_values, train_scores, 'o-', label="Train")
plt.plot(k_values, val_scores, 's-', label="Validation")
plt.xlabel("k (liczba sąsiadów)")
plt.ylabel("Accuracy")
plt.title("Overfitting vs Underfitting (KNN)")
plt.legend()
plt.grid(True)
plt.show()
# k=1: train=100% ale val niższe → OVERFIT
# Duże k: obie linie spadają → UNDERFIT

## 5. Walidacja krzyżowa (Cross-Validation)

Zamiast jednego podziału train/val, dzielimy dane na **k foldów** i trenujemy k razy:

```
Fold 1: [VAL][TRAIN][TRAIN][TRAIN][TRAIN]
Fold 2: [TRAIN][VAL][TRAIN][TRAIN][TRAIN]
Fold 3: [TRAIN][TRAIN][VAL][TRAIN][TRAIN]
Fold 4: [TRAIN][TRAIN][TRAIN][VAL][TRAIN]
Fold 5: [TRAIN][TRAIN][TRAIN][TRAIN][VAL]
```

**Wynik k-fold CV:**

$$\text{CV Score} = \frac{1}{k} \sum_{i=1}^{k} \text{Score}_i$$

$$\text{Odchylenie std} = \sqrt{\frac{1}{k-1} \sum_{i=1}^{k} (\text{Score}_i - \overline{\text{Score}})^2}$$

**Stratified k-fold** — każdy fold zachowuje proporcje klas (ważne przy niezbalansowanych danych!)

**Leave-One-Out (LOO)** — skrajny przypadek: $k = n$ (każda próbka to osobny fold). Niskie bias, ale wysokie variance i wolne.

In [ ]:
# Cross-validation w scikit-learn
knn = KNeighborsClassifier(n_neighbors=5)

# 5-fold cross-validation
scores = cross_val_score(knn, X_trainval, y_trainval, cv=5, scoring="accuracy")
print(f"Scores per fold: {scores}")
print(f"Mean: {scores.mean():.3f} ± {scores.std():.3f}")

# Porównanie różnych k z cross-validation
k_values = range(1, 21)
cv_means = []
cv_stds = []
for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    scores = cross_val_score(knn, X_trainval, y_trainval, cv=5)
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

plt.figure(figsize=(10, 5))
plt.errorbar(k_values, cv_means, yerr=cv_stds, fmt='o-', capsize=3)
plt.xlabel("k")
plt.ylabel("CV Accuracy")
plt.title("Dobór k za pomocą Cross-Validation")
plt.show()
best_k = list(k_values)[np.argmax(cv_means)]
print(f"Najlepsze k = {best_k}")

## 6. Przygotowanie danych (preprocessing)

### Standaryzacja (z-score)

$$x_{\text{std}} = \frac{x - \mu}{\sigma}$$

Po standaryzacji: średnia = 0, odchylenie standardowe = 1.

**Dlaczego?** Algorytmy oparte na odległościach (KNN, SVM) i sieci neuronowe wymagają danych w podobnych skalach.

### Normalizacja Min-Max

Alternatywnie, skalowanie do zakresu $[0, 1]$:

$$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$

Ogólniej, do zakresu $[a, b]$:

$$x_{\text{scaled}} = a + \frac{(x - x_{\min})(b - a)}{x_{\max} - x_{\min}}$$

### Robust Scaler

Odporny na outliery — używa mediany i IQR (interquartile range):

$$x_{\text{robust}} = \frac{x - \text{median}(x)}{Q_3 - Q_1}$$

### Data Leakage — krytyczne pojęcie

**Data leakage** (wyciek danych) to sytuacja, gdy informacje z danych testowych "przeciekają" do procesu uczenia.

Przykłady wycieku:
- `scaler.fit()` na **całości** danych (zamiast tylko na train)
- Feature selection na całości danych
- Imputacja braków na całości danych

**Reguła:** Wszelkie transformacje `fit()` **tylko na danych treningowych**, potem `transform()` na walidacji/teście.

$$\text{Pipeline:} \quad X_{\text{train}} \xrightarrow{\text{fit\_transform}} X'_{\text{train}} \quad | \quad X_{\text{test}} \xrightarrow{\text{transform}} X'_{\text{test}}$$

In [ ]:
# Standaryzacja
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform na TRAIN
X_val_scaled = scaler.transform(X_val)           # TYLKO transform na val!
X_test_scaled = scaler.transform(X_test)         # TYLKO transform na test!

print(f"Przed: średnie = {X_train.mean(axis=0).round(2)}")
print(f"Po:    średnie = {X_train_scaled.mean(axis=0).round(2)}")
print(f"Po:    std     = {X_train_scaled.std(axis=0).round(2)}")

# UWAGA: scaler.fit() tylko na danych treningowych!
# W przeciwnym razie mamy "data leakage" (wyciek informacji z testu)

## 7. Redukcja wymiarowości — PCA

**PCA (Principal Component Analysis)** znajduje kierunki o największej wariancji w danych i rzutuje na nie.

### Algorytm PCA krok po kroku

1. **Standaryzuj dane** — wycentruj (odejmij średnią) i opcjonalnie skaluj
2. **Oblicz macierz kowariancji:**

$$\Sigma = \frac{1}{n-1} X^T X \in \mathbb{R}^{d \times d}$$

gdzie $X \in \mathbb{R}^{n \times d}$ to wycentrowana macierz danych.

3. **Rozwiąż zagadnienie własne** (eigendecomposition):

$$\Sigma \mathbf{v}_i = \lambda_i \mathbf{v}_i$$

gdzie $\lambda_i$ — wartość własna (eigenvalue), $\mathbf{v}_i$ — wektor własny (eigenvector).

4. **Posortuj wektory własne** malejąco wg wartości własnych: $\lambda_1 \geq \lambda_2 \geq \ldots \geq \lambda_d$

5. **Weź $k$ pierwszych wektorów własnych** jako macierz projekcji $W_k = [\mathbf{v}_1 | \mathbf{v}_2 | \ldots | \mathbf{v}_k] \in \mathbb{R}^{d \times k}$

6. **Rzutuj dane:**

$$Z = X \cdot W_k \in \mathbb{R}^{n \times k}$$

### Explained Variance Ratio

Procent wariancji zachowany przez $k$-ty komponent:

$$\text{EVR}_k = \frac{\lambda_k}{\sum_{i=1}^{d} \lambda_i}$$

**Kryterium wyboru $k$**: zachowaj tyle komponentów, by suma EVR $\geq$ 95%:

$$\sum_{i=1}^{k} \text{EVR}_i \geq 0.95$$

### Interpretacja geometryczna

- PC1 = kierunek **największej wariancji** w danych
- PC2 = kierunek największej wariancji **ortogonalny** do PC1
- Każdy PCi jest kombinacją liniową oryginalnych cech: $\text{PC}_i = \sum_{j=1}^{d} v_{ij} \cdot x_j$

### Zastosowania
- Wizualizacja danych wielowymiarowych (rzut na 2D/3D)
- Usunięcie szumu (odrzucenie komponentów o małej wariancji)
- Przyspieszenie treningu (mniej wymiarów → mniej operacji)
- Dekorelacja cech (składowe główne są nieskorelowane)

### Alternatywy dla PCA
| Metoda | Typ | Zastosowanie |
|--------|-----|-------------|
| **t-SNE** | Nieliniowa | Wizualizacja 2D/3D (nie do redukcji przed modelem!) |
| **UMAP** | Nieliniowa | Wizualizacja + szybsza niż t-SNE |
| **LDA** | Liniowa, nadzorowana | Redukcja z uwzględnieniem klas |
| **Autoenkoder** | Nieliniowa, deep learning | Nieliniowa redukcja wymiarów |

In [ ]:
# PCA: redukcja 4D → 2D (Iris)
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_trainval)

print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Suma: {pca.explained_variance_ratio_.sum():.3f}")
# Ile informacji zachowaliśmy? 

# Wizualizacja
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_trainval, 
                      cmap="viridis", alpha=0.7)
plt.colorbar(scatter, label="Klasa")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% wariancji)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% wariancji)")
plt.title("Iris — PCA 2D")
plt.show()

---
---
# 🏋️ ZADANIA DO SAMODZIELNEGO ROZWIĄZANIA

---

### Zadanie 1: Metryki od zera (średnie)

Napisz **własne** implementacje (bez sklearn):
- `my_accuracy(y_true, y_pred)`
- `my_precision(y_true, y_pred)` (binarna)
- `my_recall(y_true, y_pred)` (binarna)
- `my_f1(y_true, y_pred)` (binarna)

Sprawdź wyniki z `sklearn.metrics`.

In [ ]:
# Zadanie 1: Metryki od zera
y_true = np.array([1, 1, 0, 1, 0, 0, 1, 1, 0, 1])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 1, 0, 0])

# TWÓJ KOD TUTAJ

### Zadanie 2: Algorytm k-NN od zera (trudne)

Zaimplementuj **własny** klasyfikator k-NN:
1. Oblicz odległość euklidesową do wszystkich punktów treningowych
2. Znajdź k najbliższych sąsiadów
3. Zwróć najczęstszą klasę wśród sąsiadów

Porównaj wyniki ze `sklearn.neighbors.KNeighborsClassifier`.

In [ ]:
# Zadanie 2: KNN od zera
class MyKNN:
    def __init__(self, k=5):
        self.k = k
    
    def fit(self, X, y):
        # TWÓJ KOD TUTAJ
        pass
    
    def predict(self, X):
        # TWÓJ KOD TUTAJ
        pass

# Test na Iris
my_knn = MyKNN(k=5)
my_knn.fit(X_train, y_train)
my_pred = my_knn.predict(X_val)
print(f"My KNN accuracy: {accuracy_score(y_val, my_pred):.3f}")

### Zadanie 3: Wpływ standaryzacji na KNN (średnie)

1. Wygeneruj dane ze `sklearn.datasets.make_classification` (2 cechy, 200 próbek)
2. Pomnóż jedną cechę przez 1000 (symulacja różnych skali)
3. Trenuj KNN **bez** standaryzacji → accuracy
4. Trenuj KNN **ze** standaryzacją → accuracy
5. Narysuj oba przypadki (scatter z granicami decyzji)

In [ ]:
# Zadanie 3: Wpływ standaryzacji
np.random.seed(42)
# TWÓJ KOD TUTAJ

### Zadanie 4: Klasteryzacja K-Means od zera (trudne)

Zaimplementuj algorytm K-Means:
1. Losowo zainicjuj k centroidów
2. Przypisz każdy punkt do najbliższego centroidu
3. Przelicz centroidy jako średnią przypisanych punktów
4. Powtarzaj aż centroidy się ustabilizują

Wizualizuj wynik na 2D danych.

In [ ]:
# Zadanie 4: K-Means od zera
from sklearn.datasets import make_blobs

X_blobs, y_blobs = make_blobs(n_samples=300, centers=3, random_state=42)

def my_kmeans(X, k, max_iter=100):
    # TWÓJ KOD TUTAJ
    pass

# labels, centroids = my_kmeans(X_blobs, k=3)
# Wizualizacja...

### Zadanie 5: ROC i AUC (trudne)

1. Wygeneruj binarny zbiór danych (`make_classification`)
2. Trenuj 3 modele: KNN (k=1), KNN (k=10), KNN (k=50)
3. Dla każdego modelu oblicz `predict_proba` na zbiorze testowym
4. Narysuj krzywe ROC wszystkich 3 modeli na jednym wykresie
5. Oblicz AUC dla każdego

**ROC (Receiver Operating Characteristic):** oś Y = True Positive Rate (Recall), oś X = False Positive Rate.

In [ ]:
# Zadanie 5: ROC i AUC
np.random.seed(42)
# TWÓJ KOD TUTAJ

---

## 🏆 Dodatek OAI: Ćwiczenia w stylu olimpiadowym

### Kontekst olimpiadowy
Metryki i ewaluacja to **absolutny fundament** każdego zadania OAI. W każdym zadaniu ocena jest automatyczna i oparta na konkretnej metryce. Przykłady z poprzednich edycji:

- **Balanced Accuracy** → Zaburzenia EKG (II OAI), Zmiany semantyczne (III OAI)
- **F1 macro** → Klasyfikacja wieloetykietowa (III OAI)
- **ROC AUC** → Halucynacje LLM (II OAI)
- **PSNR** → Odszumianie obrazów (II OAI), Inpainting (II OAI)
- **Accuracy** → Większość zadań klasyfikacyjnych

### Ćwiczenie OAI-5A: Funkcja punktująca (wzorzec olimpiadowy)

Napisz funkcję `compute_score(metric_value)` wzorowaną na III OAI (zmiany semantyczne):
- Balanced Accuracy ≤ 0.7 → 0 punktów
- Balanced Accuracy ≥ 0.87 → 100 punktów
- Pomiędzy: skalowanie liniowe

### Ćwiczenie OAI-5B: Porównanie modeli pod kątem czasu

Na olimpiadzie masz **limit czasowy** (zwykle 1-15 minut). Wytrenuj 3 modele na zbiorze Iris (KNN, SVM, Random Forest), zmierz czas treningu + predykcji, i wybierz najlepszy kompromis jakość/czas.

Użyj `time.time()` do pomiaru.

### Ćwiczenie OAI-5C: Cross-validation z ograniczeniami

Zaimplementuj 5-fold CV na dowolnym zbiorze, ale z ograniczeniem: cały pipeline (trening + ewaluacja) musi zmieścić się w 30 sekundach. Raportuj balanced accuracy i czas.

In [ ]:
import numpy as np
import time
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

# === Ćwiczenie OAI-5A: Funkcja punktująca ===
def compute_score(bal_acc: float, threshold_low=0.7, threshold_high=0.87) -> int:
    """
    Wzorzec z III OAI (zmiany semantyczne).
    Skalowanie liniowe między progami.
    """
    if bal_acc <= threshold_low:
        return 0
    elif bal_acc >= threshold_high:
        return 100
    else:
        return int(round(100 * (bal_acc - threshold_low) / (threshold_high - threshold_low)))

# Test
for acc in [0.5, 0.7, 0.75, 0.80, 0.87, 0.95]:
    print(f"  Balanced Accuracy = {acc:.2f} → {compute_score(acc)} pkt")

# === Ćwiczenie OAI-5B: Porównanie modeli z limitem czasu ===
print("\n=== Porównanie modeli (jakość vs czas) ===")
X, y = make_classification(n_samples=5000, n_features=20, n_classes=3, 
                           n_informative=10, random_state=42)

models = {
    'KNN(k=5)': KNeighborsClassifier(n_neighbors=5),
    'SVM(rbf)': SVC(kernel='rbf', random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42)
}

for name, model in models.items():
    start = time.time()
    scores = cross_val_score(model, X, y, cv=5, scoring='balanced_accuracy')
    elapsed = time.time() - start
    print(f"  {name:15s} | bal_acc = {scores.mean():.4f} ± {scores.std():.4f} | czas = {elapsed:.2f}s")

# Ćwiczenie OAI-5C: Twoja kolej — zaimplementuj CV z limitem 30 sekund
# TODO: Twój kod tutaj
print("\n💡 Wskazówka OAI: Na olimpiadzie zawsze mierz czas — przekroczenie limitu = 0 punktów!")